In [24]:
import pandas as pd

# 读取持仓数据
hold = pd.read_feather("CHINAMUTUALFUNDBONDPORTFOLIO.feather")
hold.columns

Index(['OBJECT_ID', 'S_INFO_WINDCODE', 'F_PRT_ENDDATE', 'CRNCY_CODE',
       'S_INFO_BONDWINDCODE', 'F_PRT_BDVALUE', 'F_PRT_BDQUANTITY',
       'F_PRT_BDVALUETONAV', 'F_ANN_DATE', 'OPDATE', 'OPMODE', 'fund_type',
       'is_equity_fund', 'available_date'],
      dtype='object')

In [2]:
import pandas as pd

files = [
    "CHINAMUTUALFUNDBONDPORTFOLIO.feather",
    "CHINAMUTUALFUNDSTOCKPORTFOLIO.feather"
]

for file in files:

    df = pd.read_feather(file)
    # 报告期转 datetime
    df["F_PRT_ENDDATE"] = pd.to_datetime(df["F_PRT_ENDDATE"],format="%Y%m%d",errors="coerce")
    mask = ((df["F_PRT_ENDDATE"].dt.month == 6) &(df["F_PRT_ENDDATE"].dt.day == 30)) | ((df["F_PRT_ENDDATE"].dt.month == 12) &(df["F_PRT_ENDDATE"].dt.day == 31))
    df = df[mask]
    # available_date = 公示日期
    def available_date(F_PRT_ENDDATE):
        if pd.isna(F_PRT_ENDDATE):
            return pd.NaT
        
        if F_PRT_ENDDATE.month == 6:
            return pd.Timestamp(
                year=F_PRT_ENDDATE.year,
                month=8,
                day=31
            )

        elif F_PRT_ENDDATE.month == 12:
            return pd.Timestamp(
                year=F_PRT_ENDDATE.year + 1,
                month=3,
                day=31
            )
        
        return pd.NaT

    df["available_date"] = df["F_PRT_ENDDATE"].apply(available_date)

    # 保存覆盖
    df.reset_index().to_feather(file)

    print(file, "done")

    

CHINAMUTUALFUNDBONDPORTFOLIO.feather done
CHINAMUTUALFUNDSTOCKPORTFOLIO.feather done


In [5]:
import pandas as pd
import numpy as np
df = pd.read_feather(
    "CHINAMUTUALFUNDSTOCKPORTFOLIO.feather"
)
# 所有 available_date 为 NaT 的记录
na_df = df[df["available_date"].isna()].copy()

print(na_df.head())

Empty DataFrame
Columns: [index, OBJECT_ID, S_INFO_WINDCODE, F_PRT_ENDDATE, CRNCY_CODE, S_INFO_STOCKWINDCODE, F_PRT_STKVALUE, F_PRT_STKQUANTITY, F_PRT_STKVALUETONAV, F_PRT_POSSTKVALUE, F_PRT_POSSTKQUANTITY, F_PRT_POSSTKTONAV, F_PRT_PASSTKEVALUE, F_PRT_PASSTKQUANTITY, F_PRT_PASSTKTONAV, ANN_DATE, STOCK_PER, FLOAT_SHR_PER, OPDATE, OPMODE, fund_type, is_equity_fund, available_date]
Index: []

[0 rows x 23 columns]
